In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"], "figure.dpi": 110})

SLIM_DIR = "/Volumes/Lexar/mglurs-brain-sc-datasets/derived/regions"
FIG_DIR = "figures/allen_wmb"
RES_DIR = "results/allen_wmb"
NAMES = {'ACA': 'Anterior cingulate area', 'AI': 'Agranular insular area', 'AUD': 'Auditory areas', 'AUD-TEa-PERI-ECT': 'Auditory/temporal/perirhinal/ectorhinal areas', 'CB': 'Cerebellum', 'CTXsp': 'Cortical subplate', 'ENT': 'Entorhinal area', 'HIP': 'Hippocampal region', 'HY': 'Hypothalamus', 'LSX': 'Lateral septal complex', 'MB': 'Midbrain', 'MY': 'Medulla', 'OLF': 'Olfactory areas', 'P': 'Pons', 'PAL': 'Pallidum', 'RHP': 'Retrohippocampal region', 'RSP': 'Retrosplenial area', 'SSp': 'Primary somatosensory area', 'STRd': 'Striatum dorsal region', 'STRv': 'Striatum ventral region', 'TH': 'Thalamus', 'VIS': 'Visual areas', 'sAMY': 'Striatum-like amygdalar nuclei', 'MOp': 'Primary motor area', 'VIS-PTLp': 'Visual/posterior parietal  areas', 'TEa-PERI-ECT': 'Temporal/perirhinal/ectorhinal areas ', 'PL-ILA-ORB': 'Prelimbic/infralimbic/orbital areas ', 'SS-GU-VISC': 'Somatosensory/gustatory/visceral areas', 'MO-FRP': 'Somatomotor - Frontal pole'}
MIN_CELLS = 200
GRMS = [f"Grm{i}" for i in range(1, 9)]

_t20 = plt.get_cmap("tab20").colors
_all = [_t20[i] for i in range(0, 20, 2)] + [_t20[i] for i in range(1, 20, 2)] + list(plt.get_cmap("tab20b").colors)
PAL = [c for c in _all if not (abs(c[0] - c[1]) < 0.03 and abs(c[1] - c[2]) < 0.03)]

def class_num(c):
    n = pd.to_numeric(str(c)[:2], errors="coerce")
    return int(n) if pd.notna(n) else 0

def class_color(c):
    n = class_num(c)
    return PAL[(n - 1) % len(PAL)] if n > 0 else (0.5, 0.5, 0.5)

def is_neuron_class(c):
    return not (30 <= class_num(c) <= 34)

def neuron_type(c):
    c = str(c)
    if not is_neuron_class(c):
        return "non-neuronal"
    if "GABA" in c:
        return "GABAergic"
    if "Glut" in c:
        return "glutamatergic"
    return "other neuronal"

FOCUS_ROIS = ["PL-ILA-ORB", "ACA", "HIP", "CTXsp", "sAMY", "STRd", "STRv"]
PAIRS = [("Grm7", "Grm8"), ("Grm7", "Grm2"), ("Grm7", "Grm3"), ("Grm2", "Grm4")]
GROUP_GENES = {"Grm": ["Grm1", "Grm2", "Grm3", "Grm4", "Grm5", "Grm6", "Grm7", "Grm8"],
               "housekeeping": ["Gapdh", "Actb", "Eef1a1", "Tubb3", "Tuba1a", "Rpl13a", "B2m"],
               "other GPCRs": ["Gabbr1", "Gabbr2", "Gpr158", "Gprc5b", "Drd1", "Drd2", "Htr2a", "Chrm1", "Adora1", "Cnr1"],
               "long neuronal": ["Csmd1", "Nrxn1", "Nrxn3", "Cntnap2", "Ctnna2", "Ptprd", "Dlg2", "Auts2"]}

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

import pyarrow.parquet as pq
LN2 = np.log(2.0)

def lin(x):
    return np.exp2(x) - 1.0

files = sorted(glob.glob(f"{SLIM_DIR}/*_slim.parquet"))
parts = []
for f in files:
    roi = os.path.basename(f).replace("_slim.parquet", "")
    names = pq.ParquetFile(f).schema.names
    genes = [g for g in GRMS if g in names]
    d = pd.read_parquet(f, columns=["class", "subclass"] + genes)
    d = d[[is_neuron_class(c) for c in d["class"]]].copy()
    for g in genes:
        d[g] = d[g].astype("float32")
    d.insert(0, "roi", roi)
    parts.append(d)
    print(roi, len(d), "neurons", flush=True)
df = pd.concat(parts, ignore_index=True)
for g in GRMS:
    if g not in df.columns:
        df[g] = np.float32(0)
df[GRMS] = df[GRMS].fillna(0)
for c in ("roi", "class", "subclass"):
    df[c] = df[c].astype("category")
thr = {g: float(np.median(df[g][df[g] > 0])) if (df[g] > 0).any() else np.inf for g in GRMS}
print("per-receptor positivity threshold (median log2 among detected neurons):", {g: round(v, 2) for g, v in thr.items()})

MIN_UNIT = 500
npos = np.zeros(len(df), dtype=np.int8)
for g in GRMS:
    npos += (df[g].to_numpy() >= thr[g]).astype(np.int8)
df["npos"] = npos
grp = df.groupby(["roi", "subclass"], observed=True)
res = pd.DataFrame({"n_cells": grp.size(),
                    "pct_ge2": grp["npos"].apply(lambda s: 100 * (s >= 2).mean()),
                    "pct_ge3": grp["npos"].apply(lambda s: 100 * (s >= 3).mean())}).reset_index()
res["class"] = res["subclass"].map(df.drop_duplicates("subclass").set_index("subclass")["class"].astype(str))
res = res[res["n_cells"] >= MIN_UNIT].sort_values("pct_ge2", ascending=False)
os.makedirs(f"{RES_DIR}/coexpression", exist_ok=True)
res.to_csv(f"{RES_DIR}/coexpression/pct_coexpress_by_celltype.csv", index=False)
pd.Series(thr, name="log2_threshold").to_csv(f"{RES_DIR}/coexpression/receptor_thresholds.csv")
summ = pd.DataFrame([(t, int((res["pct_ge2"] >= t).sum()), len(res)) for t in (25, 50, 75)], columns=["pct_cutoff", "n_celltypes_above", "n_celltypes_tested"])
summ.to_csv(f"{RES_DIR}/coexpression/pct_coexpress_summary.csv", index=False)
print(summ.to_string(index=False))
top = res.head(10)[::-1]
fig, ax = plt.subplots(figsize=(12, 5.5))
y = np.arange(len(top))
only2 = top["pct_ge2"] - top["pct_ge3"]
ax.barh(y, only2, color=plt.get_cmap("plasma")(0.25), label="exactly 2 receptors")
ax.barh(y, top["pct_ge3"], left=only2, color=plt.get_cmap("plasma")(0.8), label=">= 3 receptors")
ax.set_yticks(y)
ax.set_yticklabels([f"{r.subclass}, {NAMES.get(r.roi, r.roi)} ({r.roi}); n = {r.n_cells:,}" for r in top.itertuples()], fontsize=9)
for yi, v in zip(y, top["pct_ge2"]):
    ax.text(v + 0.8, yi, f"{v:.0f}%", va="center", fontsize=9)
ax.set_xlim(0, max(100, top["pct_ge2"].max() + 8))
ax.set_xlabel("% of cells co-expressing 2 or more mGluRs (each receptor above its own threshold)", fontsize=10)
ax.legend(frameon=False, fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
ax.set_title("Top 10 neuronal subclass x region units by pct of cells co-expressing mGluRs in single cells (Allen WMB, 10x; units with >= " + str(MIN_UNIT) + " cells)", fontsize=11)
fig.savefig(f"{FIG_DIR}/ALL_pct_coexpress_top10.png", dpi=170, bbox_inches="tight")
plt.show()